# Artificial Intelligence
## L.EIC – 3rd Year/2nd Semester
### Exercise Sheet 1
# Solving Problems by Searching

## The Two Buckets Problem

<img src="https://qph.cf2.quoracdn.net/main-qimg-45726b16b460cae0147ae8ca245a8fb0-pjlq" width="250px" height="250px" align="right">

Two buckets of capacities **c1** (e.g. 4 liters) and **c2** (e.g. 3 liters), respectively, are initially empty. The buckets do not have any intermediate markings. The only operations you can perform are:

- Fill (completely) a bucket
- Empty a bucket.
- Pour one bucket into the other until the target one is full or the source is empty.

The aim is to determine which operations to carry out so that the first bucket contains exactly **n** liters (e.g. 2 litres).

**a)** Formulate this problem as a search problem by defining the state representation, initial state, operators (their name, preconditions, effects, and cost), and final state(s).

## State Representation (b1,b2), b1 ∈ [0,4], b2 ∈ [0,3]

### Initial State (0,0)
### Final State: (2,0), (2,1), (2,2), (2,3)
### Operators:

| Name              | Preconditions                                 | Effects                | Cost |
|-------------------|-----------------------------------------------|------------------------|------|
| empty1            | b1 > 0                                        | (0, b2)                | 1    |
| empty2            | b2 > 0                                        | (b1, 0)                | 1    |
| fill1             | b1 < 4                                        | (4, b2)                | 1    |
| fill2             | b2 < 3                                        | (b1, 3)                | 1    |
| pour12_fill2      | b1 > 0 and b2 < 3 and b1 + b2 >= 3            | (b1 - (3 - b2), 3)     | 1    |
| pour12_empty1     | b1 > 0 and b2 < 3 and b1 + b2 < 3             | (0, b1 + b2)           | 1    |
| pour21_fill1      | b2 > 0 and b1 < 4 and b1 + b2 >= 4            | (4, b2 - (4 - b1))     | 1    |
| pour21_empty2     | b2 > 0 and b1 < 4 and b1 + b2 < 4             | (b1 + b2, 0)           | 1    |

**b)** Draw a partial state space graph with some of the possible transitions. What is the state space size for this problem? Consider only reachable states. *(Note: you might want to do this on paper and pencil.)*

**Answer:** The partial state space graph was drawn on the iPad. Considering only reachable states from the initial state, the space size for this problem is **14**.

**c)** Solve the problem by hand, using a search tree. What solutions have you found?

**Answer:** The search tree was drawn on the iPad. One of the solutions was: (0,0) -> (4,0) -> (1,3) -> (1,0) -> (0,1) -> (4,1) -> (2,3)

### Buildind a computational approach to handle the problem

*(Note: this notebook suggests an implementation using Python; however, feel free to use any other programming language you like, such as Prolog ;-) .)*

To build a program to solve the buckets problem, we will implement a solution that separates the problem definition from the algorithms used to traverse the state space. This way, we can reuse our implementations of the search strategies in other problems.

#### Representing the two buckets problem as a search problem

Let's start by defining a state for the buckets problem. For that, it'll suffice to aggregate two quantities, each representing the amount of water in one of the buckets. We also define a way of printing the state.

In [3]:
class BucketState:
    c1 = 4   # capacity for bucket 1
    c2 = 3   # capacity for bucket 2

    def __init__(self, b1, b2):
        self.b1 = b1
        self.b2 = b2

    '''needed for the visited list'''
    # bool operator==(const BucketState& other) em C++
    def __eq__(self, other):
        # Verifica se other é da mesma class que o self (this em C++)
        if isinstance(other, self.__class__):
            # Cada objeto Python guarda os seus atributos num dicionário interno. Esse dicionário é: onj.__dict__
            return self.__dict__ == other.__dict__
        else:
            return False

    # Equivalente a bool operator!=(const BucketState& other) em C++. ne = not equal
    def __ne__(self, other):
        """Overrides the default implementation (unnecessary in Python 3)"""
        return not self.__eq__(other)

    # Isto é muito importante para AI Search. Serve para que objetos possam ser usados em: set; dict; listas de visited states. 
    # Necessário quando queres usar objetos como chaves de um dict ou elementos de um set. E não basta __eq__ porque set e dict funcionam com hash tables.
    def __hash__(self):
        return hash((self.b1, self.b2))
    ''' - '''

    # É o equivalente a ostream& operator<<(ostream& os, const BucketState& s) em C++. Serve para apresentar de maneira correta os valores.
    # Sem isto definido e tentar fazer print(), daria <__main__.BucketState object at 0x10a7c8f10>, ou algo semelhante. Ou seja, Python usa
    # a representação default do objeto.
    def __str__(self):
        return "(" + str(self.b1) + ", " + str(self.b2) + ")"
    

Now we define each of the operators on states:

In [16]:
# emptying the first bucket
def empty1(state):
    if state.b1 > 0:
        return BucketState(0, state.b2), 1, 'empty1'   # using unitary costs
    return None

# emptying the second bucket
def empty2(state):
    if state.b2 > 0:
        return BucketState(state.b1, 0), 1, 'empty2'
    return None




# filling the first bucket
def fill1(state):
    if state.b1 < BucketState.c1:
        return BucketState(BucketState.c1, state.b2), 1, 'fill1'
    return None

# filling the second bucket
def fill2(state):
    if state.b2 < BucketState.c2:
        return BucketState(state.b1, BucketState.c2), 1, 'fill2'
    return None




# pouring from the first to the second bucket until the target is full
def pour12_fill2(state):
    if state.b1 > 0 and state.b2 < BucketState.c2 and state.b1+state.b2 >= BucketState.c2:
        return BucketState(state.b1 - (BucketState.c2 - state.b2), BucketState.c2), 1, 'pour12_fill2'
    return None

# pouring from the first to the second bucket until the source is empty
def pour12_empty1(state):
    if state.b1 > 0 and state.b2 < BucketState.c2 and state.b1+state.b2 < BucketState.c2:
        return BucketState(0, state.b1 + state.b2), 1, 'pour12_empty1'
    return None



# pouring from the second to the first bucket until the target is full
def pour21_fill1(state):
    if state.b2 > 0 and state.b1 < BucketState.c1 and state.b1+state.b2 >= BucketState.c1:
        return BucketState(BucketState.c1, state.b2 - (BucketState.c1 - state.b1)), 1, 'pour21_fill1'
    return None




# pouring from the second to the first bucket until the source is empty
def pour21_empty2(state):
    if state.b2 > 0 and state.b1 < BucketState.c1 and state.b1+state.b2 < BucketState.c1:
        return BucketState(state.b1 + state.b2, 0), 1, 'pour21_empty2'
    return None


The following function will aggregate all states that can be generated from a given one:

In [5]:
def child_bucket_states(state):
    new_states = []
    if(empty1(state)):
        new_states.append(empty1(state))
    if(empty2(state)):
        new_states.append(empty2(state))
    if(fill1(state)):
        new_states.append(fill1(state))
    if(fill2(state)):
        new_states.append(fill2(state))
    if(pour12_fill2(state)):
        new_states.append(pour12_fill2(state))
    if(pour12_empty1(state)):
        new_states.append(pour12_empty1(state))
    if(pour21_fill1(state)):
        new_states.append(pour21_fill1(state))
    if(pour21_empty2(state)):
        new_states.append(pour21_empty2(state))
    return new_states

Play around with the state transition operators and check if they are working properly:

In [18]:
# Cria o estado inicial
s = BucketState(0, 0)
# s passa a ser (BucketState(0,3), 1)
s = fill2(s)
# O __str__ não é usado aqui porque o Python está a imprimir o tuple, não diretamente o objeto BucketState
print(s)

# Imprime uma linha em branco
print()

# Como a função devolve uma lista de tuplos (BucketState, custo), o child, _ é para ir apenas buscar o BucketState e ignorar o custo
for child, _, operator in child_bucket_states(BucketState(0, 0)):
    print(child, operator)

# empty1

print(empty1(BucketState(4,2)))   # deve funcionar
print(empty1(BucketState(0,2)))   # deve dar None


# empty2

print(empty2(BucketState(1,3)))   # deve funcionar
print(empty2(BucketState(1,0)))   # deve dar None


# fill1

print(fill1(BucketState(0,2)))    # deve funcionar
print(fill1(BucketState(4,2)))    # deve dar None


# fill2

print(fill2(BucketState(1,0)))    # deve funcionar
print(fill2(BucketState(1,3)))    # deve dar None


# pour12_fill2

print(pour12_fill2(BucketState(4,1)))   # deve dar (2,3)
print(pour12_fill2(BucketState(2,0)))   # deve dar None


# pour12_empty1

print(pour12_empty1(BucketState(2,0)))  # deve dar (0,2)
print(pour12_empty1(BucketState(4,1)))  # deve dar None


# pour21_fill1

print(pour21_fill1(BucketState(3,3)))   # deve dar (4,2)
print(pour21_fill1(BucketState(1,2)))   # deve dar None


# pour21_empty2

print(pour21_empty2(BucketState(1,2)))  # deve dar (3,0)
print(pour21_empty2(BucketState(3,3)))  # deve dar None

(<__main__.BucketState object at 0x000001F4D80FB2F0>, 1, 'fill2')

(4, 0) fill1
(0, 3) fill2
(<__main__.BucketState object at 0x000001F4D8ED48C0>, 1, 'empty1')
None
(<__main__.BucketState object at 0x000001F4D8ED4BC0>, 1, 'empty2')
None
(<__main__.BucketState object at 0x000001F4D8ED48C0>, 1, 'fill1')
None
(<__main__.BucketState object at 0x000001F4D8ED4BC0>, 1, 'fill2')
None
(<__main__.BucketState object at 0x000001F4D8ED48C0>, 1, 'pour12_fill2')
None
(<__main__.BucketState object at 0x000001F4D8ED4BC0>, 1, 'pour12_empty1')
None
(<__main__.BucketState object at 0x000001F4D8ED48C0>, 1, 'pour21_fill1')
None
(<__main__.BucketState object at 0x000001F4D8ED4BC0>, 1, 'pour21_empty2')
None


Finally, we need to define the goal condition:

In [ ]:
def goal_bucket_state(state):
    return state.b1 == 2




Test your goal condition:

In [8]:
# your code here

print(goal_bucket_state(BucketState(0,2)))
print(goal_bucket_state(BucketState(2,0)))
print(goal_bucket_state(BucketState(3,2)))
print(goal_bucket_state(BucketState(2,3)))
print(goal_bucket_state(BucketState(4,3)))


False
True
False
True
False


#### Implementing search algorithms

Let us start by defining an appropriate structure to represent a node in a search tree. Each tree node will include:
- a state of the problem
- a link to its parent (to allow traveling from a leaf node towards the root of the tree)
- a list of child nodes

In [6]:
# A generic definition of a tree node holding a state of the problem
class TreeNode:
    def __init__(self, state, parent=None, operator=None):
        self.state = state
        self.parent = parent
        self.children = []
        self.operator = operator

    def add_child(self, child_node):
        self.children.append(child_node)
        child_node.parent = self

##### Breadth-first search

Based on this structure, we can now implement breadth-first search. Note that we want the implementation to be independent of the problem at hand (in this case, the two buckets problem).

In [4]:
from collections import deque

def breadth_first_search(initial_state, goal_state_func, operators_func):
    root = TreeNode(initial_state)   # create the root node in the search tree
    queue = deque([root])   # initialize the queue to store the nodes
    visited = set([initial_state]) # to prevent revisiting a node

    while queue:
        node = queue.popleft()   # get first element in the queue

        if goal_state_func(node.state):   # check goal state
            return node

        for state, _, operator in operators_func(node.state):   # go through next states
            if state not in visited:
                # create tree node with the new state
                childNode = TreeNode(state, operator=operator) # no need to pass parent because add_child() function automatically places the parent value in childNode
            
                # link child node to its parent in the tree
                node.add_child(childNode)

                # enqueue the child node
                queue.append(childNode)

                # mark as visited
                visited.add(state)

    return None

We can now use this function to actually perform a breadth-first search on the buckets problem: we pass it the initial state, our goal condition function, and the function for obtaining child states.

In [25]:
goal = breadth_first_search(BucketState(0,0),
                            goal_bucket_state,
                            child_bucket_states)
print(goal.state)

(2, 3)


In order to print the actual steps from the initial state to the last, we can take advantage of each node's link to its parent.

In [8]:
def print_solution(node):
    path = []
    
    while node:
        path.append(node)
        node = node.parent

    path.reverse()

    for node in path:
        if node.operator:
            print(node.operator)
        print(node.state)

    return

Now we can print the solution:

In [34]:
print_solution(goal)

(0, 0)
fill1
(4, 0)
pour12_fill2
(1, 3)
empty2
(1, 0)
pour12_empty1
(0, 1)
fill1
(4, 1)
pour12_fill2
(2, 3)


If we need a description for each of the employed operators, we could have each operation function return also such a description, and modify the TreeNode class so that each node also includes a description of the edge to get there. We leave that as an exercise after class.

Already implemented.

##### Depth-first search

Implement depth-first search (again, in a manner that is independent of the problem at hand). You can start from your breadth-first search implementation and with minor changes get an implementation for depth-first search.

In [41]:
def depth_first_search(initial_state, goal_state_func, operators_func, visited=None, node=None):
    if visited is None:
        visited = set()
    
    if node is None:
        node = TreeNode(initial_state)

    visited.add(node.state) # to prevent revisiting a node

    if goal_state_func(node.state):   # check goal state
        return node

    for state, _, operator in operators_func(node.state):   # go through next states
        if state not in visited:
            # create tree node with the new state
            childNode = TreeNode(state, operator=operator) # no need to pass parent because add_child() function automatically places the parent value in childNode
            
            # link child node to its parent in the tree
            node.add_child(childNode)

            result = depth_first_search(childNode.state, goal_state_func, operators_func, visited, childNode)

            if result:
                return result

    return None

Test it on the two buckets problem.

In [42]:
solution = depth_first_search(BucketState(0,0), goal_bucket_state, child_bucket_states)

if solution:
    print("Solution found:")
    print_solution(solution)
else:
    print("No solution found")

Solution found:
(0, 0)
fill1
(4, 0)
fill2
(4, 3)
empty1
(0, 3)
pour21_empty2
(3, 0)
fill2
(3, 3)
pour21_fill1
(4, 2)
empty1
(0, 2)
pour21_empty2
(2, 0)


If you are unable to get a solution, think about it: depth-first search is not a complete search method, and one of the reasons for that is if the state space contains cycles. As such, you need to make sure you avoid entering into a cycle by keeping a visited nodes list or set and checking that list whenever you generate a new state.

##### Depth-limited search

Another way to make it work is to impose a depth limit to the problem. Implement depth-limited search.

In [43]:
def depth_limited_search(initial_state, goal_state_func, operators_func, depth_limit, depth=0, visited=None, node=None):
    if visited is None:
        visited = set()

    if node is None:
        node = TreeNode(initial_state)

    if depth > depth_limit:
        return None
    
    visited.add(node.state)

    if goal_state_func(node.state):
        return node

    for state, _, operator in operators_func(node.state):
        if state not in visited:
            childNode = TreeNode(state, operator=operator)

            node.add_child(childNode)

            result = depth_limited_search(childNode.state, goal_state_func, operators_func, depth_limit, depth + 1, visited, childNode)

            if result:
                return result

    return None

Test it on the two buckets problem.

In [52]:
solution = depth_limited_search(BucketState(0,0), goal_bucket_state, child_bucket_states, 6)

if solution:
    print("Solution found:")
    print_solution(solution)
else:
    print("No solution found within depth limit")

print(depth_limited_search(BucketState(0,0), goal_bucket_state, child_bucket_states, 0))
print(depth_limited_search(BucketState(0,0), goal_bucket_state, child_bucket_states, 1))
print(depth_limited_search(BucketState(0,0), goal_bucket_state, child_bucket_states, 2))
print(depth_limited_search(BucketState(0,0), goal_bucket_state, child_bucket_states, 3))
print(depth_limited_search(BucketState(0,0), goal_bucket_state, child_bucket_states, 4))
print(depth_limited_search(BucketState(0,0), goal_bucket_state, child_bucket_states, 5))
print(depth_limited_search(BucketState(0,0), goal_bucket_state, child_bucket_states, 6))





Solution found:
(0, 0)
fill1
(4, 0)
pour12_fill2
(1, 3)
empty2
(1, 0)
pour12_empty1
(0, 1)
fill1
(4, 1)
pour12_fill2
(2, 3)
None
None
None
None
None
None


##### Iterative deepening search

Based on depth-limited, you can easily implement iterative-deepening search.

In [55]:
def iterative_deepening_search(initial_state, goal_state_func, operators_func):
    depth_limit = 0

    while True:
        result = depth_limited_search(initial_state, goal_state_func, operators_func, depth_limit)

        if result:
            return result

        depth_limit += 1

Again, test it on the two buckets problem.

In [56]:
solution = iterative_deepening_search(BucketState(0,0), goal_bucket_state, child_bucket_states)

if solution:
    print("Solution found:")
    print_solution(solution)
else:
    print("No solution found")






Solution found:
(0, 0)
fill1
(4, 0)
pour12_fill2
(1, 3)
empty2
(1, 0)
pour12_empty1
(0, 1)
fill1
(4, 1)
pour12_fill2
(2, 3)


## The Missionaries and Cannibals Problem

<img src="https://allfish24.files.wordpress.com/2016/09/gambar-1.jpg">

Three missionaries and three cannibals are on one of the banks of the river with a boat that only takes one or two people. The boat cannot travel the river alone.

The goal is to find a way to get the six to the other bank of the river without ever having more cannibals than missionaries on one of the banks (even at the instant they leave/join the boat) during the process.

**a)** Formulate this problem as a search problem by defining the state representation, initial state, operators (their name, preconditions, effects, and cost), and final state(s).

**Answer:**

State representation: `(M, C, B)`, where `M` and `C` are the numbers of missionaries and cannibals on the initial bank, and `B ∈ {dir, esq}` is the boat position.

Initial state: `(3, 3, dir)`

Goal state: `(0, 0, esq)`

A state is valid only if both banks are safe: on each bank, missionaries are never outnumbered by cannibals unless there are no missionaries on that bank.

Operators (all with cost 1):

| Name | Preconditions | Effects | Cost |
|---|---|---|---|
| take1M_to_esq | B = dir and M > 0 | (M - 1, C, esq) | 1 |
| take2M_to_esq | B = dir and M > 1 | (M - 2, C, esq) | 1 |
| take1C_to_esq | B = dir and C > 0 | (M, C - 1, esq) | 1 |
| take2C_to_esq | B = dir and C > 1 | (M, C - 2, esq) | 1 |
| take1M_1C_to_esq | B = dir and M > 0 and C > 0 | (M - 1, C - 1, esq) | 1 |
| take1M_to_dir | B = esq and (3 - M) > 0 | (M + 1, C, dir) | 1 |
| take2M_to_dir | B = esq and (3 - M) > 1 | (M + 2, C, dir) | 1 |
| take1C_to_dir | B = esq and (3 - C) > 0 | (M, C + 1, dir) | 1 |
| take2C_to_dir | B = esq and (3 - C) > 1 | (M, C + 2, dir) | 1 |
| take1M_1C_to_dir | B = esq and (3 - M) > 0 and (3 - C) > 0 | (M + 1, C + 1, dir) | 1 |

Only transitions that preserve the safety invariant are allowed.

**b)** Draw a partial state space graph with some of the possible transitions. What is the state space size for this problem? Consider only reachable states. *(Note: you might want to do this on paper and pencil.)*

**Answer:** The partial state space graph was drawn on paper/iPad. Considering only reachable legal states from `(3, 3, dir)`, the state space has **16** states.

**c)** Solve the problem by hand, using a search tree. What solutions have you found?

**d)** Make your own implementation of the problem as a search problem and take advantage of the implemented search algorithms to find solutions.

In [2]:
class MisCanState:

    def __init__(self, mis, can, boat):
        self.mis = mis   # number of missionaries on the initial bank
        self.can = can   # number of cannibals on the initial bank
        self.boat = boat   # boat on the initial bank?

    '''needed for the visited list'''
    def __eq__(self, other):
        if isinstance(other, self.__class__):
            return self.__dict__ == other.__dict__
        else:
            return False

    def __ne__(self, other):
        """Overrides the default implementation (unnecessary in Python 3)"""
        return not self.__eq__(other)

    def __hash__(self):
        return hash((self.mis, self.can, self.boat))
    ''' - '''

    def __str__(self):
        return "(" + str(self.mis) + ", " + str(self.can) + ", " + str(self.boat) + ")"


def is_valid_miscan_state(state):
    left_safe = state.mis == 0 or state.mis >= state.can
    right_mis = 3 - state.mis
    right_can = 3 - state.can
    right_safe = right_mis == 0 or right_mis >= right_can
    return 0 <= state.mis <= 3 and 0 <= state.can <= 3 and left_safe and right_safe


def transition_miscan(state, missionaries, cannibals, operator_name):
    if state.boat == 'dir':
        if state.mis < missionaries or state.can < cannibals:
            return None
        next_state = MisCanState(state.mis - missionaries, state.can - cannibals, 'esq')
    elif state.boat == 'esq':
        if (3 - state.mis) < missionaries or (3 - state.can) < cannibals:
            return None
        next_state = MisCanState(state.mis + missionaries, state.can + cannibals, 'dir')
    else:
        return None

    if is_valid_miscan_state(next_state):
        return next_state, 1, operator_name
    return None


def take1M_to_esq(state):
    if state.boat != 'dir':
        return None
    return transition_miscan(state, 1, 0, 'take1M_to_esq')


def take2M_to_esq(state):
    if state.boat != 'dir':
        return None
    return transition_miscan(state, 2, 0, 'take2M_to_esq')


def take1C_to_esq(state):
    if state.boat != 'dir':
        return None
    return transition_miscan(state, 0, 1, 'take1C_to_esq')


def take2C_to_esq(state):
    if state.boat != 'dir':
        return None
    return transition_miscan(state, 0, 2, 'take2C_to_esq')


def take1M_1C_to_esq(state):
    if state.boat != 'dir':
        return None
    return transition_miscan(state, 1, 1, 'take1M_1C_to_esq')


def take1M_to_dir(state):
    if state.boat != 'esq':
        return None
    return transition_miscan(state, 1, 0, 'take1M_to_dir')


def take2M_to_dir(state):
    if state.boat != 'esq':
        return None
    return transition_miscan(state, 2, 0, 'take2M_to_dir')


def take1C_to_dir(state):
    if state.boat != 'esq':
        return None
    return transition_miscan(state, 0, 1, 'take1C_to_dir')


def take2C_to_dir(state):
    if state.boat != 'esq':
        return None
    return transition_miscan(state, 0, 2, 'take2C_to_dir')


def take1M_1C_to_dir(state):
    if state.boat != 'esq':
        return None
    return transition_miscan(state, 1, 1, 'take1M_1C_to_dir')

In [9]:
def child_miscan_states(state):
    new_states = []

    for operator in [
        take1M_to_esq,
        take2M_to_esq,
        take1C_to_esq,
        take2C_to_esq,
        take1M_1C_to_esq,
        take1M_to_dir,
        take2M_to_dir,
        take1C_to_dir,
        take2C_to_dir,
        take1M_1C_to_dir
    ]:
        result = operator(state)
        if result:
            new_states.append(result)

    return new_states


def goal_miscan_state(state):
    return state.mis == 0 and state.can == 0 and state.boat == 'esq'


def reachable_miscan_states(initial_state):
    visited = set([initial_state])
    queue = [initial_state]

    while queue:
        current = queue.pop(0)
        for child_state, _, _ in child_miscan_states(current):
            if child_state not in visited:
                visited.add(child_state)
                queue.append(child_state)

    return visited


initial_state = MisCanState(3, 3, 'dir')
reachable_states = reachable_miscan_states(initial_state)
print('Reachable states:', len(reachable_states))
print(sorted(str(state) for state in reachable_states))

solution = breadth_first_search(initial_state, goal_miscan_state, child_miscan_states)
print()
print('BFS solution:')
print_solution(solution)

Reachable states: 16
['(0, 0, esq)', '(0, 1, dir)', '(0, 1, esq)', '(0, 2, dir)', '(0, 2, esq)', '(0, 3, dir)', '(1, 1, dir)', '(1, 1, esq)', '(2, 2, dir)', '(2, 2, esq)', '(3, 0, esq)', '(3, 1, dir)', '(3, 1, esq)', '(3, 2, dir)', '(3, 2, esq)', '(3, 3, dir)']

BFS solution:
(3, 3, dir)
take2C_to_esq
(3, 1, esq)
take1C_to_dir
(3, 2, dir)
take2C_to_esq
(3, 0, esq)
take1C_to_dir
(3, 1, dir)
take2M_to_esq
(1, 1, esq)
take1M_1C_to_dir
(2, 2, dir)
take2M_to_esq
(0, 2, esq)
take1C_to_dir
(0, 3, dir)
take2C_to_esq
(0, 1, esq)
take1M_to_dir
(1, 1, dir)
take1M_1C_to_esq
(0, 0, esq)
